# Where does the recency mean lose fantasy points?

For each priced Six Nations round, select one squad using the 180-day recency mean and one using actual points. Both use the project's scorer, budget, team limits and slot rules. The recency mean trains on all available competitions before the first match of the round.

Use the position table to find recurring gaps and the player table to see which picks produced them. This is a comparison between two full squads. The hindsight squad can spend its budget and team slots differently, so a position's gap is the difference in contributions, rather than the points available from an isolated swap.

## Data and seasons

Install the repository in the notebook's Python environment with `pip install -e .`. The default data directory is `data/rebuild` in the repository. It needs `matches.parquet`, `player_performances.parquet` and a prepared `six_nations_<year>_fixtures.parquet` for each requested year.

A roster can cover fewer than five rounds. The notebook reports which rounds were skipped. Use complete match lineups and prices within each round you want to compare.

In [ ]:
from math import isclose

import pandas as pd
from IPython.display import display

from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.data.fantasy_fixtures import fixture_path, tournament_rounds
from six_nations_fantasy.data.load import DATA_DIR
from six_nations_fantasy.evaluation.pipeline import fit_predict
from six_nations_fantasy.evaluation.selection import prediction_lineup, scored_player_fixtures
from six_nations_fantasy.evaluation.squads import select_squad, squad_solver
from six_nations_fantasy.ilp import POSITION_COUNTS
from six_nations_fantasy.models import RecencyWeightedMean
from six_nations_fantasy.score import POINTS_PER_UNIT, simple_score

data_dir = DATA_DIR
years = [2026]  # Add 2025 if its prepared roster is available.


In [ ]:
dataset = load_dataset(data_dir)
solver = squad_solver()
model = RecencyWeightedMean(scorer=simple_score, half_life_days=180)
round_keys = ["year", "round_number"]
squads = []
round_results = []

for year in years:
    roster_path = fixture_path(data_dir, year)
    if not roster_path.exists():
        raise FileNotFoundError(f"Prepare the {year} fantasy roster first: {roster_path}")
    roster = pd.read_parquet(roster_path)
    for round_number, schedule in tournament_rounds(dataset.matches, year):
        round_roster = roster.loc[roster["round"].eq(round_number)]
        if round_roster.empty:
            print(f"{year} round {round_number}: no roster, skipped")
            continue

        match_ids = set(schedule["match_id"])
        played_round = dataset.select_matches(match_ids)
        if set(round_roster["match_id"]) != match_ids:
            raise ValueError(f"{year} round {round_number}: roster and schedule cover different matches")
        if set(played_round.performances["match_id"]) != match_ids:
            raise ValueError(f"{year} round {round_number}: missing match performances")

        training = dataset.matches_before(schedule["date"].min())
        scored = scored_player_fixtures(played_round, round_number, roster)
        predicted = fit_predict(model, training, prediction_lineup(scored))
        candidates = scored.assign(predicted=predicted).dropna(subset=["cost"])

        recency = select_squad(candidates, "predicted", solver)
        optimum = select_squad(candidates, "actual", solver)
        selected_xv = recency.loc[~recency["supersub"]]
        captain_gain = selected_xv["actual"].max() - selected_xv.loc[
            selected_xv["captain"], "actual"
        ].iloc[0]
        round_results.append({
            "year": year,
            "round_number": round_number,
            "recency_points": recency["actual_total"].sum(),
            "optimum_points": optimum["actual_total"].sum(),
            "captain_gain_in_same_xv": captain_gain,
        })
        for label, squad in [("Recency mean", recency), ("Optimum", optimum)]:
            squads.append(squad.assign(year=year, round_number=round_number, squad=label))

if not round_results:
    raise ValueError("No rounds with a prepared fantasy roster were found")

selected = pd.concat(squads, ignore_index=True)
round_totals = pd.DataFrame(round_results).set_index(round_keys).sort_index()
round_totals["gap"] = round_totals["optimum_points"] - round_totals["recency_points"]
display(round_totals.round(1))


## Gap by position and bonus

Count each fantasy XV player once at their listed fantasy position. Add the captain's extra copy of their points and the supersub's entire multiplied score as separate rows. A positive gap means the hindsight squad scored more in that category. The signed gaps add up to the full squad gap in each round.

Use **gap per slot per round** to compare positions with different numbers of slots. The median and number of rounds behind help distinguish a repeated gap from a single costly round. The captain bonus gap can reflect different XVs. **Captain gain in same XV** above isolates the extra points available by changing only the recency squad's captain with hindsight.

In [ ]:
keys = ["match_id", "player_id"]
try_counts = dataset.select_matches(selected["match_id"]).performances[keys + ["tries"]]
selected_with_tries = selected.merge(try_counts, on=keys, how="left", validate="many_to_one")
# The scorer gives zero points to an announced player with no recorded appearance.
selected_with_tries["tries"] = selected_with_tries["tries"].fillna(0)
xv = selected_with_tries.loc[~selected_with_tries["supersub"]]
captains = xv.loc[xv["captain"]]
supersubs = selected_with_tries.loc[selected_with_tries["supersub"]]
supersub_multiplier = supersubs["is_sub"].map({True: 3.0, False: 0.5})

contributions = pd.concat([
    xv.assign(category=xv["nominal_pos"], points=xv["actual"], multiplier=1),
    captains.assign(category="Captain bonus", points=captains["actual"], multiplier=1),
    supersubs.assign(
        category="Supersub", points=supersubs["actual_total"], multiplier=supersub_multiplier
    ),
], ignore_index=True)
contributions["try_points"] = contributions["tries"] * POINTS_PER_UNIT["tries"] * contributions["multiplier"]
slots = {**POSITION_COUNTS, "Captain bonus": 1, "Supersub": 1}
round_gaps = (
    contributions.groupby([*round_keys, "category", "squad"])["points"]
    .sum().unstack("squad", fill_value=0)
)
round_gaps["gap"] = round_gaps["Optimum"] - round_gaps["Recency mean"]
gap_by_round = round_gaps["gap"].unstack("category").reindex(columns=list(slots), fill_value=0)
for key, gap in gap_by_round.sum(axis=1).items():
    assert isclose(gap, round_totals.loc[key, "gap"], abs_tol=1e-8), key

summary = pd.DataFrame({
    "total_gap": gap_by_round.sum(),
    "median_round_gap": gap_by_round.median(),
    "rounds_behind": gap_by_round.gt(0).sum(),
    "rounds_compared": len(gap_by_round),
})
summary["gap_per_slot_per_round"] = gap_by_round.mean() / pd.Series(slots)
display(summary.sort_values("gap_per_slot_per_round", ascending=False).round(1))
display(gap_by_round.round(1))


## How much of the gap comes from tries?

For the same two squads, subtract the points awarded directly for scoring tries. Try assists and conversions stay in **other points**. A try by the fantasy captain counts once in their position and once in the captain bonus. A supersub's try uses the supersub multiplier.

The signed gaps are hindsight minus recency. A negative try gap means the recency squad scored more try points. The round table shows whether the try gap was concentrated in one round. Each row reconciles to the original points gap for those picks.

In [ ]:
try_totals = (
    contributions.groupby([*round_keys, "category", "squad"])["try_points"]
    .sum().unstack("squad", fill_value=0)
)
try_gaps = round_gaps[["gap"]].copy()
try_gaps["try_gap"] = try_totals["Optimum"] - try_totals["Recency mean"]
try_gaps["other_gap"] = try_gaps["gap"] - try_gaps["try_gap"]

gap_columns = ["gap", "try_gap", "other_gap"]
by_category = try_gaps.groupby("category")[gap_columns].sum()
display(by_category.sort_values("gap", ascending=False).round(1))
display(try_gaps[gap_columns].round(1))
by_round = try_gaps.groupby(level=round_keys)[gap_columns].sum()
display(by_round.round(1))


## Starting XV or bench in the real match?

Compare points and number of picks by announced role. The fantasy XV can include players announced on the real bench. Captain bonuses and supersub points appear on their own rows. A role gap can arise because the squads picked different numbers of players in that role or because they chose different players within it.

In [ ]:
role_labels = {False: "starter", True: "bench"}
situations = pd.concat([
    xv.assign(situation="Fantasy XV · " + xv["is_sub"].map(role_labels), points=xv["actual"]),
    captains.assign(
        situation="Captain bonus · " + captains["is_sub"].map(role_labels),
        points=captains["actual"],
    ),
    supersubs.assign(
        situation="Supersub · " + supersubs["is_sub"].map(role_labels),
        points=supersubs["actual_total"],
    ),
], ignore_index=True)

role_summary = (
    situations.groupby(["situation", "squad"])["points"].agg(["sum", "size"])
    .unstack("squad", fill_value=0)
)
role_table = pd.DataFrame({
    "recency_picks": role_summary[("size", "Recency mean")],
    "optimum_picks": role_summary[("size", "Optimum")],
    "recency_points": role_summary[("sum", "Recency mean")],
    "optimum_points": role_summary[("sum", "Optimum")],
})
role_table["gap"] = role_table["optimum_points"] - role_table["recency_points"]
assert isclose(role_table["gap"].sum(), round_totals["gap"].sum(), abs_tol=1e-8)
display(role_table.sort_values("gap", ascending=False).round(1))


## Inspect the picks behind a gap

The default shows the largest position or bonus gap in any round. Change the year, round or category to inspect another. Read the forecast beside the actual minutes and tries to see what happened. Minutes and tries are recorded outcomes used only for this inspection. Missing minutes or tries may indicate a late withdrawal or an unavailable record.

In [ ]:
review_year, review_round, review_category = round_gaps["gap"].idxmax()

picks = contributions.loc[
    contributions["year"].eq(review_year)
    & contributions["round_number"].eq(review_round)
    & contributions["category"].eq(review_category)
].copy()

keys = ["match_id", "player_id"]
available_stats = ["minutes_played"] if "minutes_played" in dataset.performances else []
if available_stats:
    performances = dataset.select_matches(picks["match_id"]).performances
    picks = picks.merge(performances[keys + available_stats], on=keys, how="left", validate="many_to_one")
picks["announced_role"] = picks["is_sub"].map(role_labels)
print(f"{review_year} round {review_round} · {review_category}")
display(
    picks.sort_values(["squad", "points"], ascending=[True, False])[
        ["squad", "name", "team", "announced_role", "cost", "predicted", "actual",
         "points", "tries", "try_points"]
        + available_stats
    ].round(1)
)
